# CMT — ARCADE Inpainting Pipeline

Train the CMT inpainting model on vessel-masked coronary angiography X-rays.

---
⚠️ **Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── 2. Clone repo ─────────────────────────────────────────────────────────
!git clone https://github.com/C0d3Crush/arcade-xray-inpainting.git
%cd /content/arcade-xray-inpainting

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── 4. Mount Google Drive & unzip ARCADE dataset ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path to where arcade.zip lives in your Drive
ARCADE_ZIP = '/content/drive/MyDrive/arcade.zip'

!unzip -q -o "$ARCADE_ZIP" -d /content/arcade-xray-inpainting/
!find arcade -name '*.json' | head -5

---
## Generate Random Masks

In [ ]:
# ── 5. Generate random vessel-shaped masks ────────────────────────────────
!python generate_random_masks.py \
    --annotations arcade/syntax/train/annotations/train.json \
    --images      arcade/syntax/train/images \
    --output      arcade/syntax/train/random_masks \
    --n_shapes    5 \
    --seed        42

!python generate_random_masks.py \
    --annotations arcade/syntax/val/annotations/val.json \
    --images      arcade/syntax/val/images \
    --output      arcade/syntax/val/random_masks \
    --n_shapes    5 \
    --seed        42

---
## Train CMT Inpainting Model

In [ ]:
# ── 6. Train CMT ──────────────────────────────────────────────────────────
!python train.py \
    --train_img  arcade/syntax/train/images \
    --train_ann  arcade/syntax/train/annotations/train.json \
    --train_mask arcade/syntax/train/random_masks \
    --val_img    arcade/syntax/val/images \
    --val_ann    arcade/syntax/val/annotations/val.json \
    --val_mask   arcade/syntax/val/random_masks \
    --input_size 256 \
    --epochs     100 \
    --batch_size 16 \
    --device     cuda

---
## Download Checkpoint

In [ ]:
# ── 7. Download best checkpoint ───────────────────────────────────────────
from google.colab import files
files.download('checkpoints/best.pth')

---
## Nach dem Download — lokal einrichten

```bash
mv ~/Downloads/best.pth checkpoints/
```

Inference lokal:
```bash
python demo.py \
  --ckpt checkpoints/best.pth \
  --img_path ./samples/test_img \
  --mask_path ./samples/test_mask \
  --output_path ./samples/results \
  --device cpu
```